In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from collections import Counter
import re
import time
import os
import json

# ====== 설정 ======
stopwords = {"the", "and", "for", "with", "that", "this", "from", "are", "was", "has", "have", "but", "our"}
base_year = 2025
start_week = 45  # 시작 주차
max_weeks = 52   # 연도별 최대 주차

# ====== 웹 드라이버 실행 ======
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

year = base_year
week = start_week

while True:
    week_url = f"https://huggingface.co/papers/week/{year}-W{week:02d}"
    print(f"\n🔹 Crawling {week_url} ...")
    driver.get(week_url)

    # 페이지 존재 여부 확인
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "article h3 a"))
        )
    except:
        print(f"❌ Not found or no articles: {week_url} -> 크롤링 종료")
        break

    # 폴더 생성
    folder = f"{year}-W{week:02d}"
    os.makedirs(folder, exist_ok=True)

    # 파일명 시작 번호: 연도(2자리) + 주차(2자리) + 001
    file_index = int(str(year)[-2:] + f"{week:02d}" + "001")

    # 아티클 링크 추출
    articles = driver.find_elements(By.CSS_SELECTOR, "article h3 a")
    article_urls = [a.get_attribute("href") for a in articles]

    for link in article_urls:
        driver.get(link)

        # 논문 제목
        try:
            paper_name = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "h1"))
            ).text.strip()
        except:
            paper_name = "Unknown_Title"

        # Abstract
        page_content = ""
        try:
            abstract_div = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "div.pb-8.pr-4.md\\:pr-16 > div"))
            )
            ps = abstract_div.find_elements(By.TAG_NAME, "p")
            if ps:
                page_content = "\n".join([p.text.strip() for p in ps])
            else:
                page_content = abstract_div.text.strip()
        except:
            page_content = ""

        # Upvote
        try:
            upvote_elem = WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.CSS_SELECTOR,
                    "section.pt-8 div.hidden.flex-wrap.items-start.gap-2.md\\:flex > div > div > a > div > div"
                ))
            )
            upvote_text = upvote_elem.text.strip()
            upvote_match = re.search(r"\d+", upvote_text)
            upvote = int(upvote_match.group()) if upvote_match else 0
        except:
            upvote = 0

        # GitHub 링크
        try:
            github_url = driver.find_element(By.XPATH, "//a[contains(@href,'github.com')]").get_attribute("href")
        except:
            github_url = ""

        huggingface_url = link

        # 태그 추출 (abstract 단어 상위 3개)
        words = re.findall(r'\b\w+\b', page_content.lower())
        filtered = [w for w in words if w not in stopwords and len(w) > 2]
        counter = Counter(filtered)
        tags = [tag for tag, _ in counter.most_common(3)]
        while len(tags) < 3:
            tags.append("")

        # metadata dictionary
        MetaData = {
            "papername": paper_name,
            "github_url": github_url,
            "huggingface_url": huggingface_url,
            "upvote": upvote,
            "tag1": tags[0],
            "tag2": tags[1],
            "tag3": tags[2]
        }

        # 파일명 및 경로
        doc_name = f"doc{file_index}.txt"
        file_path = os.path.join(folder, doc_name)

        # txt 파일 작성
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(f"page_content:\n{page_content}\n\nMetaData:\n")
            f.write(json.dumps(MetaData, ensure_ascii=False, indent=4))

        print(f"✅ Saved {file_path}")
        file_index += 1

    # 다음 주차로 증가
    week += 1
    if week > max_weeks:
        week = 1
        year += 1

driver.quit()
print("모든 아티클 저장 완료!")



🔹 Crawling https://huggingface.co/papers/week/2025-W45 ...
✅ Saved 2025-W45\doc2545001.txt
✅ Saved 2025-W45\doc2545002.txt
✅ Saved 2025-W45\doc2545003.txt
✅ Saved 2025-W45\doc2545004.txt
✅ Saved 2025-W45\doc2545005.txt
✅ Saved 2025-W45\doc2545006.txt
✅ Saved 2025-W45\doc2545007.txt
✅ Saved 2025-W45\doc2545008.txt
✅ Saved 2025-W45\doc2545009.txt
✅ Saved 2025-W45\doc2545010.txt
✅ Saved 2025-W45\doc2545011.txt
✅ Saved 2025-W45\doc2545012.txt
✅ Saved 2025-W45\doc2545013.txt
✅ Saved 2025-W45\doc2545014.txt
✅ Saved 2025-W45\doc2545015.txt
✅ Saved 2025-W45\doc2545016.txt
✅ Saved 2025-W45\doc2545017.txt
✅ Saved 2025-W45\doc2545018.txt
✅ Saved 2025-W45\doc2545019.txt
✅ Saved 2025-W45\doc2545020.txt
✅ Saved 2025-W45\doc2545021.txt
✅ Saved 2025-W45\doc2545022.txt
✅ Saved 2025-W45\doc2545023.txt
✅ Saved 2025-W45\doc2545024.txt
✅ Saved 2025-W45\doc2545025.txt
✅ Saved 2025-W45\doc2545026.txt
✅ Saved 2025-W45\doc2545027.txt
✅ Saved 2025-W45\doc2545028.txt
✅ Saved 2025-W45\doc2545029.txt
✅ Saved 2025

==================================================================

In [2]:
import os
import json

# 확인할 폴더들 (예: 2025-W45, 2025-W46 등)
folders = [f for f in os.listdir() if os.path.isdir(f) and f.startswith("2025-W")]

for folder in folders:
    print(f"\n🔹 Checking folder: {folder}")
    empty_count = 0

    # 폴더 안의 txt 파일 확인
    for filename in os.listdir(folder):
        if filename.endswith(".txt"):
            file_path = os.path.join(folder, filename)
            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()
                
                # page_content 부분 추출
                if "page_content:\n" in content:
                    page_content = content.split("page_content:\n")[1].split("\n\nMetaData:")[0].strip()
                    if not page_content:
                        print(f"⚠️ Empty page_content: {file_path}")
                        empty_count += 1
                else:
                    print(f"❌ page_content not found: {file_path}")
                    empty_count += 1

    print(f"✅ Total empty page_content in {folder}: {empty_count}")



🔹 Checking folder: 2025-W45
✅ Total empty page_content in 2025-W45: 0

🔹 Checking folder: 2025-W46
✅ Total empty page_content in 2025-W46: 0


In [3]:
import os
import json
import re

# 체크할 최상위 폴더 (예: 현재 폴더)
base_dir = "."

# 폴더 패턴: 2025-W45, 2025-W46 ...
week_folders = [f for f in os.listdir(base_dir) if re.match(r"\d{4}-W\d{2}", f)]

for folder in week_folders:
    folder_path = os.path.join(base_dir, folder)
    txt_files = [f for f in os.listdir(folder_path) if f.endswith(".txt")]

    for txt_file in txt_files:
        file_path = os.path.join(folder_path, txt_file)
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()

        # metadata 부분 추출
        meta_match = re.search(r"MetaData:\s*(\{.*\})", content, re.DOTALL)
        if not meta_match:
            print(f"⚠️ MetaData not found: {file_path}")
            continue

        try:
            metadata = json.loads(meta_match.group(1))
        except:
            print(f"⚠️ MetaData JSON parse error: {file_path}")
            continue

        # 빈 값 체크
        empty_fields = [k for k, v in metadata.items() if v == "" or v is None]
        if empty_fields:
            print(f"⚠️ Empty MetaData fields in {file_path}: {empty_fields}")


⚠️ Empty MetaData fields in .\2025-W45\doc2545006.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545008.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545013.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545014.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545015.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545016.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545017.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545020.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545021.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545023.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545026.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545028.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545030.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545031.txt: ['github_url']
⚠️ Empty MetaData fi

In [10]:
import os
from langchain_community.document_loaders import TextLoader

# 폴더 경로 지정
folder_path = "2025-W45===="

all_docs = []

print("🔍 TXT 파일 로드 중...\n")

# 폴더 내 모든 txt 파일 반복
for filename in os.listdir(folder_path):
    if filename.endswith(".txt"):
        file_path = os.path.join(folder_path, filename)

        # 파일 로드
        loader = TextLoader(file_path, encoding="utf-8")
        docs = loader.load()

        # 로드된 문서 저장
        all_docs.extend(docs)

        # 각 문서 확인
        for doc in docs:
            print(f"\n======================")
            print(f"📄 파일명: {filename}")
            print(f"metadata: {doc.metadata}")
            print("\n📌 page_content(앞 300자 미리보기):")
            preview = doc.page_content[:50].strip()
            print(preview if preview else "⚠️ 내용 없음 (EMPTY)")
            print("======================\n")


print(f"\n총 로드된 문서 수: {len(all_docs)}")


🔍 TXT 파일 로드 중...


📄 파일명: doc2545001.txt
metadata: {'source': '2025-W45====\\doc2545001.txt'}

📌 page_content(앞 300자 미리보기):
page_content:
The "Thinking with Video" paradigm e


📄 파일명: doc2545002.txt
metadata: {'source': '2025-W45====\\doc2545002.txt'}

📌 page_content(앞 300자 미리보기):
page_content:
Diffusion language models outperform


📄 파일명: doc2545003.txt
metadata: {'source': '2025-W45====\\doc2545003.txt'}

📌 page_content(앞 300자 미리보기):
page_content:
VCode introduces a benchmark for gen


📄 파일명: doc2545004.txt
metadata: {'source': '2025-W45====\\doc2545004.txt'}

📌 page_content(앞 300자 미리보기):
page_content:
V-Thinker, a multimodal reasoning as


📄 파일명: doc2545005.txt
metadata: {'source': '2025-W45====\\doc2545005.txt'}

📌 page_content(앞 300자 미리보기):
page_content:
Systematic study reveals that naive


📄 파일명: doc2545006.txt
metadata: {'source': '2025-W45====\\doc2545006.txt'}

📌 page_content(앞 300자 미리보기):
page_content:
Ling 2.0, a reasoning-oriented langu


📄 파일명: doc2545007.txt
metadata: {'

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from collections import Counter
import re, time, os, json, random

# ====== 설정 ======
base_year = 2025
start_week = 45
wait_time = 10               # 페이지 렌더링 대기시간
max_retry_per_article = 5    # 아티클 단위 재시도
retry_click = 5              # 오른쪽 버튼 클릭 재시도
stopwords = {"the", "and", "for", "with", "that", "this", "from", "are", "was", "has", "have", "but", "our"}

# ====== 웹 드라이버 실행 ======
options = webdriver.ChromeOptions()
options.add_argument("--headless=new") # 브라우저를 **화면 없이** 실행. 즉, 실제 창이 뜨지 않고 백그라운드에서 동작
options.add_argument("--disable-gpu") # GPU 하드웨어 가속을 사용하지 않음. 주로 헤드리스 모드에서 안정성 위해 추가
options.add_argument("--window-size=1920,1080")  # 브라우저 가상 창 크기를 지정. 화면 크기 기반 렌더링이나 스크린샷 필요 시 중요
options.add_argument(f"--user-agent=Mozilla/5.0") # 웹 서버에 "브라우저 요청"임을 알리는 User-Agent 설정

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# ====== 초기 주차 URL ======
week = start_week
week_url = f"https://huggingface.co/papers/week/{base_year}-W{week:02d}"
driver.get(week_url)

# ====== 파일명 시작 ======
file_index = int(str(base_year)[-2:] + f"{week:02d}" + "001")

while True:
    print(f"\n🔹 Crawling {week_url} ...")
    folder = f"{base_year}-W{week:02d}"
    os.makedirs(folder, exist_ok=True)

    # 랜덤 대기
    time.sleep(random.uniform(8, 15))

    # 아티클 링크 추출
    try:
        WebDriverWait(driver, wait_time).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "article h3 a"))
        )
        articles = driver.find_elements(By.CSS_SELECTOR, "article h3 a")
        article_urls = [a.get_attribute("href") for a in articles]
    except:
        print(f"❌ No articles found on {week_url}. 스킵")
        break

    for link in article_urls:
        for attempt in range(1, max_retry_per_article+1):
            try:
                driver.get(link)
                time.sleep(random.uniform(8, 12))  # 랜덤 대기

                # 논문 제목
                try:
                    paper_name = WebDriverWait(driver, wait_time).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "h1"))
                    ).text.strip()
                except:
                    paper_name = "Unknown_Title"

                # Abstract
                page_content = ""
                try:
                    abstract_div = WebDriverWait(driver, wait_time).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "div.pb-8.pr-4.md\\:pr-16 > div"))
                    )
                    ps = abstract_div.find_elements(By.TAG_NAME, "p")
                    page_content = "\n".join([p.text.strip() for p in ps]) if ps else abstract_div.text.strip()
                except:
                    page_content = ""

                # Upvote
                try:
                    upvote_elem = WebDriverWait(driver, wait_time).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR,
                            "section.pt-8 div.hidden.flex-wrap.items-start.gap-2.md\\:flex > div > div > a > div > div"
                        ))
                    )
                    upvote_match = re.search(r"\d+", upvote_elem.text.strip())
                    upvote = int(upvote_match.group()) if upvote_match else 0
                except:
                    upvote = 0

                # GitHub 링크
                try:
                    github_url = driver.find_element(By.XPATH, "//a[contains(@href,'github.com')]").get_attribute("href")
                except:
                    github_url = ""

                huggingface_url = link

                # 태그 추출
                words = re.findall(r'\b\w+\b', page_content.lower())
                filtered = [w for w in words if w not in stopwords and len(w) > 2]
                counter = Counter(filtered)
                tags = [tag for tag, _ in counter.most_common(3)]
                while len(tags) < 3:
                    tags.append("")

                # metadata dictionary
                MetaData = {
                    "papername": paper_name,
                    "github_url": github_url,
                    "huggingface_url": huggingface_url,
                    "upvote": upvote,
                    "tag1": tags[0],
                    "tag2": tags[1],
                    "tag3": tags[2]
                }

                # 파일명 및 경로
                doc_name = f"doc{file_index}.txt"
                file_path = os.path.join(folder, doc_name)

                # txt 파일 작성
                with open(file_path, "w", encoding="utf-8") as f:
                    f.write(f"page_content:\n{page_content}\n\nMetaData:\n")
                    f.write(json.dumps(MetaData, ensure_ascii=False, indent=4))

                print(f"✅ Saved {file_path}")
                file_index += 1
                break  # 성공하면 재시도 종료

            except Exception as e:
                print(f"⚠️ 아티클 로딩 실패, 재시도 {attempt}/{max_retry_per_article}")
                time.sleep(10)  # 재시도 전 대기

    # ====== 다음 주 오른쪽 화살표 클릭 ======
    clicked = False
    for attempt in range(retry_click):
        try:
            next_btn = WebDriverWait(driver, wait_time).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR,
                    "body > div.flex.min-h-dvh.flex-col > main > div:nth-child(2) > section div.ml-6.flex.items-center.overflow-hidden a:nth-child(3)"
                ))
            )
            next_btn.click()
            time.sleep(random.uniform(8, 15))
            week_url = driver.current_url
            week += 1
            file_index = int(str(base_year)[-2:] + f"{week:02d}" + "001")
            clicked = True
            break
        except:
            print(f"⚠️ 오른쪽 버튼 클릭 실패, 재시도 {attempt+1}/{retry_click}")
            time.sleep(3)

    if not clicked:
        print("➡ 더 이상 오른쪽 버튼 없음 -> 최신 주차 도달, 크롤링 종료")
        break

driver.quit()
print("모든 아티클 저장 완료!")



🔹 Crawling https://huggingface.co/papers/week/2025-W45 ...
✅ Saved 2025-W45\doc2545001.txt
✅ Saved 2025-W45\doc2545002.txt
✅ Saved 2025-W45\doc2545003.txt
✅ Saved 2025-W45\doc2545004.txt
✅ Saved 2025-W45\doc2545005.txt
✅ Saved 2025-W45\doc2545006.txt
✅ Saved 2025-W45\doc2545007.txt
✅ Saved 2025-W45\doc2545008.txt
✅ Saved 2025-W45\doc2545009.txt
✅ Saved 2025-W45\doc2545010.txt
✅ Saved 2025-W45\doc2545011.txt
✅ Saved 2025-W45\doc2545012.txt
✅ Saved 2025-W45\doc2545013.txt
✅ Saved 2025-W45\doc2545014.txt
✅ Saved 2025-W45\doc2545015.txt
✅ Saved 2025-W45\doc2545016.txt
✅ Saved 2025-W45\doc2545017.txt
✅ Saved 2025-W45\doc2545018.txt
✅ Saved 2025-W45\doc2545019.txt
✅ Saved 2025-W45\doc2545020.txt
✅ Saved 2025-W45\doc2545021.txt
✅ Saved 2025-W45\doc2545022.txt
✅ Saved 2025-W45\doc2545023.txt
✅ Saved 2025-W45\doc2545024.txt
✅ Saved 2025-W45\doc2545025.txt
✅ Saved 2025-W45\doc2545026.txt
✅ Saved 2025-W45\doc2545027.txt
✅ Saved 2025-W45\doc2545028.txt
✅ Saved 2025-W45\doc2545029.txt
✅ Saved 2025

## JSON으로 크롤링 : 기본 불용어

In [ ]:
## JSON 으로 크롤링
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from collections import Counter
import re, time, os, json, random

# ====== 설정 ======
base_year = 2025
start_week = 45
wait_time = 7
max_retry_per_article = 4
retry_click = 6
stopwords = {"the", "and", "for", "with", "that", "this", "from", "are", "was", "has", "have", "but", "our"}

# ====== 웹 드라이버 실행 ======
options = webdriver.ChromeOptions()
# options.add_argument("--headless=new")   # 속도 느릴 경우 headless 제거하면 오히려 스무스함
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# ====== 초기 URL ======
week = start_week
week_url = f"https://huggingface.co/papers/week/{base_year}-W{week:02d}"
driver.get(week_url)


# ====== 파일 번호 ======
file_index = int(str(base_year)[-2:] + f"{week:02d}" + "001")


while True:
    print(f"\n🔹 Crawling {week_url} ...")

    folder = f"{base_year}-W{week:02d}"
    os.makedirs(folder, exist_ok=True)

    # 페이지 로딩 안정화
    time.sleep(random.uniform(3, 6))

    # 아티클 링크 추출
    try:
        WebDriverWait(driver, wait_time).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "article h3 a"))
        )
        articles = driver.find_elements(By.CSS_SELECTOR, "article h3 a")
        article_urls = [a.get_attribute("href") for a in articles]
    except:
        print("❌ No articles found. 종료")
        break

    for link in article_urls:
        for attempt in range(1, max_retry_per_article + 1):
            try:
                driver.get(link)
                time.sleep(random.uniform(3, 6))

                # 제목
                try:
                    paper_name = WebDriverWait(driver, wait_time).until(
                        EC.presence_of_element_located((By.TAG_NAME, "h1"))
                    ).text.strip()
                except:
                    paper_name = "Unknown_Title"

                # Abstract
                try:
                    abstract_div = WebDriverWait(driver, wait_time).until(
                        EC.presence_of_element_located(
                            (By.CSS_SELECTOR, "div.pb-8.pr-4.md\\:pr-16 > div")
                        )
                    )
                    ps = abstract_div.find_elements(By.TAG_NAME, "p")
                    page_content = "\n".join([p.text.strip() for p in ps]) if ps else abstract_div.text.strip()
                except:
                    page_content = ""

                # Upvote
                try:
                    upvote_elem = WebDriverWait(driver, wait_time).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR,
                            "section.pt-8 div.hidden.flex-wrap.items-start.gap-2.md\\:flex a div div"
                        ))
                    )
                    upvote_match = re.search(r"\d+", upvote_elem.text.strip())
                    upvote = int(upvote_match.group()) if upvote_match else 0
                except:
                    upvote = 0

                # GitHub 링크
                try:
                    github_url = driver.find_element(By.XPATH, "//a[contains(@href,'github.com')]").get_attribute("href")
                except:
                    github_url = ""

                huggingface_url = link

                # 태그 추출
                words = re.findall(r'\b\w+\b', page_content.lower())
                filtered = [w for w in words if w not in stopwords and len(w) > 2]
                counter = Counter(filtered)
                tags = [tag for tag, _ in counter.most_common(3)]
                while len(tags) < 3:
                    tags.append("")

                # JSON 구조
                json_data = {
                    "content": page_content,
                    "metadata": {
                        "paper_name": paper_name,
                        "github_url": github_url,
                        "huggingface_url": huggingface_url,
                        "upvote": upvote,
                        "tags": tags
                    }
                }

                # 저장
                doc_name = f"doc{file_index}.json"
                file_path = os.path.join(folder, doc_name)

                with open(file_path, "w", encoding="utf-8") as f:
                    json.dump(json_data, f, ensure_ascii=False, indent=4)

                print(f"✅ Saved {file_path}")
                file_index += 1
                break

            except:
                print(f"⚠️ 재시도 {attempt}/{max_retry_per_article}")
                time.sleep(3)

    # ====== 다음 주 버튼 클릭 ======
    clicked = False
    for attempt in range(retry_click):
        try:
            next_btn = WebDriverWait(driver, wait_time).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "a[aria-label='Next week']"))
            )
            next_btn.click()
            time.sleep(random.uniform(3, 6))

            week_url = driver.current_url
            week += 1
            file_index = int(str(base_year)[-2:] + f"{week:02d}" + "001")
            clicked = True
            break
        except:
            print(f"⚠️ Next 버튼 클릭 실패 {attempt+1}/{retry_click}")
            time.sleep(2)

    if not clicked:
        print("➡ 더 이상 다음 주 없음 → 종료")
        break


driver.quit()
print("\n🎉 모든 아티클 저장 완료!")



🔹 Crawling https://huggingface.co/papers/week/2025-W45 ...
✅ Saved 2025-W45\doc2545001.json
✅ Saved 2025-W45\doc2545002.json
✅ Saved 2025-W45\doc2545003.json
✅ Saved 2025-W45\doc2545004.json
✅ Saved 2025-W45\doc2545005.json
✅ Saved 2025-W45\doc2545006.json
✅ Saved 2025-W45\doc2545007.json
✅ Saved 2025-W45\doc2545008.json
✅ Saved 2025-W45\doc2545009.json
✅ Saved 2025-W45\doc2545010.json
✅ Saved 2025-W45\doc2545011.json
✅ Saved 2025-W45\doc2545012.json
✅ Saved 2025-W45\doc2545013.json
✅ Saved 2025-W45\doc2545014.json
✅ Saved 2025-W45\doc2545015.json
✅ Saved 2025-W45\doc2545016.json
✅ Saved 2025-W45\doc2545017.json
✅ Saved 2025-W45\doc2545018.json
✅ Saved 2025-W45\doc2545019.json
✅ Saved 2025-W45\doc2545020.json
✅ Saved 2025-W45\doc2545021.json
✅ Saved 2025-W45\doc2545022.json
✅ Saved 2025-W45\doc2545023.json
✅ Saved 2025-W45\doc2545024.json
✅ Saved 2025-W45\doc2545025.json
✅ Saved 2025-W45\doc2545026.json
✅ Saved 2025-W45\doc2545027.json
✅ Saved 2025-W45\doc2545028.json
✅ Saved 2025-W45

## 저장된 JSON 파일 확인

In [ ]:
import os
import json
from langchain_core.documents import Document  # 변경된 부분

folder_path = "2025-W45"
all_docs = []

for filename in os.listdir(folder_path):
    if filename.endswith(".json"):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

            # Document 객체 생성
            doc = Document(
                page_content=data.get("content", ""),
                metadata={
                    "papername": data["metadata"].get("papername", ""),
                    "github_url": data["metadata"].get("github_url", ""),
                    "huggingface_url": data["metadata"].get("huggingface_url", ""),
                    "upvote": data["metadata"].get("upvote", 0),
                    "tag1": data["metadata"].get("tag1", ""),
                    "tag2": data["metadata"].get("tag2", ""),
                    "tag3": data["metadata"].get("tag3", "")
    }
            )
            all_docs.append(doc)

# 테스트 출력
for doc in all_docs:
    print("======================")
    print("page_content preview:", doc.page_content[:100])
    print("papername:", doc.metadata.get("papername", ""))
    print("github_url:", doc.metadata.get("github_url", ""))
    print("huggingface_url:", doc.metadata.get("huggingface_url", ""))
    print("upvote:", doc.metadata.get("upvote", 0))
    print("tag1:", doc.metadata.get("tag1", ""))
    print("tag2:", doc.metadata.get("tag2", ""))
    print("tag3:", doc.metadata.get("tag3", ""))
    print("======================")


page_content preview: The "Thinking with Video" paradigm enhances multimodal reasoning by integrating video generation mod
papername: Thinking with Video: Video Generation as a Promising Multimodal Reasoning Paradigm
github_url: https://github.com/tongjingqi/Thinking-with-Video
huggingface_url: https://huggingface.co/papers/2511.04570
upvote: 207
tag1: video
tag2: thinking
tag3: tasks
page_content preview: Diffusion language models outperform autoregressive models in low-data settings due to any-order mod
papername: Diffusion Language Models are Super Data Learners
github_url: https://github.com/JinjieNi/dlms-are-super-data-learners
huggingface_url: https://huggingface.co/papers/2511.03276
upvote: 121
tag1: models
tag2: data
tag3: settings
page_content preview: VCode introduces a benchmark for generating SVG code from images to preserve symbolic meaning, highl
papername: VCode: a Multimodal Coding Benchmark with SVG as Symbolic Visual Representation
github_url: https://github.com/CSU-J

## 저장된 JSON 파일 비어있는것 있는지 확인

In [4]:
import os
import json

# 확인할 폴더들 (예: 2025-W45, 2025-W46 등)
folders = [f for f in os.listdir() if os.path.isdir(f) and f.startswith("2025-W")]

for folder in folders:
    print(f"\n🔹 Checking folder: {folder}")
    empty_count = 0

    # 폴더 안의 JSON 파일 확인
    for filename in os.listdir(folder):
        if filename.endswith(".json"):
            file_path = os.path.join(folder, filename)
            with open(file_path, "r", encoding="utf-8") as f:
                try:
                    data = json.load(f)
                    page_content = data.get("content", "").strip()
                    if not page_content:
                        print(f"⚠️ Empty content: {file_path}")
                        empty_count += 1
                except Exception as e:
                    print(f"❌ Failed to load JSON: {file_path} ({e})")
                    empty_count += 1

    print(f"✅ Total empty page_content in {folder}: {empty_count}")



🔹 Checking folder: 2025-W45
✅ Total empty page_content in 2025-W45: 0

🔹 Checking folder: 2025-W45====
✅ Total empty page_content in 2025-W45====: 0

🔹 Checking folder: 2025-W46
✅ Total empty page_content in 2025-W46: 0


## JSON 크롤링

In [1]:
import os
import time
import random
import re
import json
from collections import Counter
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import logging
from datetime import datetime


# ====== NLTK 불용어 적용 ======
import nltk
from nltk.corpus import stopwords as nltk_stopwords
nltk.download('stopwords')

# 기본 영어 불용어
stopwords = set(nltk_stopwords.words("english"))

# 논문/논문사이트 특화 불용어 추가
extra_stopwords = {
    "introduction", "method", "result", "figure", "table",
    "dataset", "experiment", "paper", "approach", "related", "work"
}
stopwords.update(extra_stopwords)

# ====== 설정 ======
base_year = 2025
start_week = 45
wait_time = 7
max_retry_per_article = 4
retry_click = 6

# ====== 로깅 설정 ======
# ====== 로깅 파일 이름 생성 ======
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
log_week_str = f"{base_year}-W{start_week:02d}"
log_file = f"crawling_{log_week_str}_{current_time}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_file, mode='w', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logging.info(f"🚀 크롤링 시작 — 로그파일: {log_file}")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_file, mode='w', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logging.info("🚀 크롤링 시작")

# ====== 웹 드라이버 실행 ======
options = webdriver.ChromeOptions()
# options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("user-agent=Mozilla/5.0")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# ====== 초기 주차 URL ======
week = start_week
week_url = f"https://huggingface.co/papers/week/{base_year}-W{week:02d}"
file_index = int(str(base_year)[-2:] + f"{week:02d}" + "001")

# ====== 통계 ======
total_articles = 0
success_count = 0
fail_count = 0

# ====== 크롤링 루프 ======
while True:
    logging.info(f"🔹 Crawling week URL: {week_url}")
    folder = f"{base_year}-W{week:02d}"
    os.makedirs(folder, exist_ok=True)
    time.sleep(random.uniform(3, 6))

    # 아티클 링크 추출
    try:
        driver.get(week_url)
        WebDriverWait(driver, wait_time).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "article h3 a"))
        )
        articles = driver.find_elements(By.CSS_SELECTOR, "article h3 a")
        article_urls = [a.get_attribute("href") for a in articles]
        logging.info(f"📝 {len(article_urls)} articles found")
    except Exception as e:
        logging.error(f"❌ No articles found or page error: {e}")
        break

    total_articles += len(article_urls)

    # 각 아티클 크롤링
    for link in article_urls:
        article_success = False
        for attempt in range(1, max_retry_per_article + 1):
            try:
                driver.get(link)
                time.sleep(random.uniform(3, 6))

                # 제목
                try:
                    paper_name = WebDriverWait(driver, wait_time).until(
                        EC.presence_of_element_located((By.TAG_NAME, "h1"))
                    ).text.strip()
                except:
                    paper_name = "Unknown_Title"

                # Abstract
                try:
                    abstract_div = WebDriverWait(driver, wait_time).until(
                        EC.presence_of_element_located(
                            (By.CSS_SELECTOR, "div.pb-8.pr-4.md\\:pr-16 > div")
                        )
                    )
                    ps = abstract_div.find_elements(By.TAG_NAME, "p")
                    page_content = "\n".join([p.text.strip() for p in ps]) if ps else abstract_div.text.strip()
                except:
                    page_content = ""

                # Upvote
                try:
                    upvote_elem = WebDriverWait(driver, wait_time).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR,
                            "section.pt-8 div.hidden.flex-wrap.items-start.gap-2.md\\:flex a div div"
                        ))
                    )
                    upvote_match = re.search(r"\d+", upvote_elem.text.strip())
                    upvote = int(upvote_match.group()) if upvote_match else 0
                except:
                    upvote = 0

                # GitHub 링크
                try:
                    github_url = driver.find_element(By.XPATH, "//a[contains(@href,'github.com')]").get_attribute("href")
                except:
                    github_url = ""

                huggingface_url = link

                # 태그 추출
                words = re.findall(r'\b\w+\b', page_content.lower())
                filtered = [w for w in words if w not in stopwords and len(w) > 2]
                counter = Counter(filtered)
                tags = [tag for tag, _ in counter.most_common(3)]
                while len(tags) < 3:
                    tags.append("")

                # JSON 구조
                json_data = {
                    "content": page_content,
                    "metadata": {
                        "paper_name": paper_name,
                        "github_url": github_url,
                        "huggingface_url": huggingface_url,
                        "upvote": upvote,
                        "tags": tags
                    }
                }

                # 저장
                doc_name = f"doc{file_index}.json"
                file_path = os.path.join(folder, doc_name)
                with open(file_path, "w", encoding="utf-8") as f:
                    json.dump(json_data, f, ensure_ascii=False, indent=4)

                logging.info(f"✅ Saved {file_path}")
                file_index += 1
                success_count += 1
                article_success = True
                break

            except Exception as e:
                logging.warning(f"⚠️ Retry {attempt}/{max_retry_per_article} failed for {link}, error: {e}")
                time.sleep(3)

        if not article_success:
            logging.error(f"❌ Failed to crawl article: {link}")
            fail_count += 1

    # ====== 다음 주 버튼 클릭 (XPath 사용) ======
    clicked = False
    for attempt in range(retry_click):
        try:
            driver.get(week_url)  # 주차 리스트 페이지로 이동
            next_btn = WebDriverWait(driver, wait_time).until(
                EC.element_to_be_clickable((By.XPATH, "/html/body/div[1]/main/div[2]/section/div[1]/div[4]/div/div[2]/a[2]"))
            )
            next_btn.click()
            time.sleep(random.uniform(3, 6))
            week += 1
            week_url = driver.current_url
            file_index = int(str(base_year)[-2:] + f"{week:02d}" + "001")
            clicked = True
            logging.info(f"➡ Moving to next week: {week_url}")
            break
        except:
            logging.warning(f"⚠️ Next button click attempt {attempt+1}/{retry_click} failed")
            time.sleep(2)

    if not clicked:
        logging.info("➡ No more weeks. Crawling finished.")
        break

driver.quit()

# ====== 최종 통계 ======
logging.info("🎉 크롤링 완료!")
logging.info(f"총 아티클 수: {total_articles}")
logging.info(f"성공: {success_count}")
logging.info(f"실패: {fail_count}")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
2025-12-03 14:58:35,658 [INFO] 🚀 크롤링 시작 — 로그파일: crawling_2025-W45_20251203_145835.log
2025-12-03 14:58:35,658 [INFO] 🚀 크롤링 시작
2025-12-03 14:58:35,658 [INFO] ====== WebDriver manager ======
2025-12-03 14:58:36,798 [INFO] Get LATEST chromedriver version for google-chrome
2025-12-03 14:58:36,846 [INFO] Get LATEST chromedriver version for google-chrome
2025-12-03 14:58:36,893 [INFO] Driver [C:\Users\playdata2\.wdm\drivers\chromedriver\win64\142.0.7444.175\chromedriver-win32/chromedriver.exe] found in cache
2025-12-03 14:58:38,246 [INFO] 🔹 Crawling week URL: https://huggingface.co/papers/week/2025-W45
2025-12-03 14:58:43,831 [INFO] 📝 105 articles found
2025-12-03 14:58:49,945 [INFO] ✅ Saved 2025-W45\doc2545001.json
2025-12-03 14:58:55,647 [INFO] ✅ Saved 2025-W45\doc2545002.json
2025-12-03 14:59:02,381 [INFO] ✅ Saved 2025-W45\doc2

## JSON 크롤링 파일 빈 것 확인

In [2]:
import os
import json

# 확인할 폴더들 (예: 2025-W45, 2025-W46 등)
folders = [f for f in os.listdir() if os.path.isdir(f) and f.startswith("2025-W")]

for folder in folders:
    print(f"\n🔹 Checking folder: {folder}")
    empty_content_count = 0
    incomplete_metadata_count = 0

    # 폴더 안의 JSON 파일 확인
    for filename in os.listdir(folder):
        if filename.endswith(".json"):
            file_path = os.path.join(folder, filename)
            with open(file_path, "r", encoding="utf-8") as f:
                try:
                    data = json.load(f)
                    # content 확인
                    page_content = data.get("content", "").strip()
                    if not page_content:
                        print(f"⚠️ Empty content: {file_path}")
                        empty_content_count += 1

                    # metadata 확인
                    metadata = data.get("metadata", {})
                    missing_fields = []
                    for key in ["paper_name", "github_url", "huggingface_url", "upvote", "tags"]:
                        if key not in metadata or metadata[key] in [None, "", []]:
                            missing_fields.append(key)
                    if missing_fields:
                        print(f"⚠️ Incomplete metadata ({', '.join(missing_fields)}): {file_path}")
                        incomplete_metadata_count += 1

                except Exception as e:
                    print(f"❌ Failed to load JSON: {file_path} ({e})")
                    empty_content_count += 1
                    incomplete_metadata_count += 1

    print(f"✅ Total empty content in {folder}: {empty_content_count}")
    print(f"✅ Total incomplete metadata in {folder}: {incomplete_metadata_count}")



🔹 Checking folder: 2025-W45
⚠️ Incomplete metadata (github_url): 2025-W45\doc2545006.json
⚠️ Incomplete metadata (github_url): 2025-W45\doc2545008.json
⚠️ Incomplete metadata (github_url): 2025-W45\doc2545013.json
⚠️ Incomplete metadata (github_url): 2025-W45\doc2545014.json
⚠️ Incomplete metadata (github_url): 2025-W45\doc2545015.json
⚠️ Incomplete metadata (github_url): 2025-W45\doc2545016.json
⚠️ Incomplete metadata (github_url): 2025-W45\doc2545017.json
⚠️ Incomplete metadata (github_url): 2025-W45\doc2545020.json
⚠️ Incomplete metadata (github_url): 2025-W45\doc2545021.json
⚠️ Incomplete metadata (github_url): 2025-W45\doc2545023.json
⚠️ Incomplete metadata (github_url): 2025-W45\doc2545026.json
⚠️ Incomplete metadata (github_url): 2025-W45\doc2545028.json
⚠️ Incomplete metadata (github_url): 2025-W45\doc2545030.json
⚠️ Incomplete metadata (github_url): 2025-W45\doc2545031.json
⚠️ Incomplete metadata (github_url): 2025-W45\doc2545032.json
⚠️ Incomplete metadata (github_url): 2025